In [2]:
##lora fine tuning demo
#This shows the standard training pipeline in a praactical way
import os
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("lofty_finetune_dataset.csv")

In [5]:
df

,input,output
0,Describe project summary,Lofty is a local-first AI assistant project wi...
1,Describe project summary please,Lofty is a local-first AI assistant project wi...
2,Describe project summary now,Lofty is a local-first AI assistant project wi...
3,Describe project summary in simple terms,Lofty is a local-first AI assistant project wi...
4,Describe project summary in one sentence,Lofty is a local-first AI assistant project wi...
...,...,...
6195,architecture description variant 205,"Lofty is built with a FastAPI API layer, an as..."
6196,backend framework description variant 205,"Lofty uses FastAPI to serve endpoints, handle ..."
6197,memory management description variant 205,Lofty stores session messages and user prefere...
6198,model integration description variant 205,Lofty integrates with Ollama for local model i...


In [6]:
df.isnull().sum()

input     0
output    0
dtype: int64

In [7]:
df.columns.tolist()

['input', 'output']

In [8]:
df

,input,output
0,Describe project summary,Lofty is a local-first AI assistant project wi...
1,Describe project summary please,Lofty is a local-first AI assistant project wi...
2,Describe project summary now,Lofty is a local-first AI assistant project wi...
3,Describe project summary in simple terms,Lofty is a local-first AI assistant project wi...
4,Describe project summary in one sentence,Lofty is a local-first AI assistant project wi...
...,...,...
6195,architecture description variant 205,"Lofty is built with a FastAPI API layer, an as..."
6196,backend framework description variant 205,"Lofty uses FastAPI to serve endpoints, handle ..."
6197,memory management description variant 205,Lofty stores session messages and user prefere...
6198,model integration description variant 205,Lofty integrates with Ollama for local model i...


In [15]:
df2 = df
df2['fulltext'] = "### Input:"+ df['input']+"\n### Output:"+df['output']


In [19]:
df2[['fulltext']]

,fulltext
0,### Input:Describe project summary\n### Output...
1,### Input:Describe project summary please\n###...
2,### Input:Describe project summary now\n### Ou...
3,### Input:Describe project summary in simple t...
4,### Input:Describe project summary in one sent...
...,...
6195,### Input:architecture description variant 205...
6196,### Input:backend framework description varian...
6197,### Input:memory management description varian...
6198,### Input:model integration description varian...


In [28]:
df

,input,output,fulltext
0,Describe project summary,Lofty is a local-first AI assistant project wi...,### Input:Describe project summary\n### Output...
1,Describe project summary please,Lofty is a local-first AI assistant project wi...,### Input:Describe project summary please\n###...
2,Describe project summary now,Lofty is a local-first AI assistant project wi...,### Input:Describe project summary now\n### Ou...
3,Describe project summary in simple terms,Lofty is a local-first AI assistant project wi...,### Input:Describe project summary in simple t...
4,Describe project summary in one sentence,Lofty is a local-first AI assistant project wi...,### Input:Describe project summary in one sent...
...,...,...,...
6195,architecture description variant 205,"Lofty is built with a FastAPI API layer, an as...",### Input:architecture description variant 205...
6196,backend framework description variant 205,"Lofty uses FastAPI to serve endpoints, handle ...",### Input:backend framework description varian...
6197,memory management description variant 205,Lofty stores session messages and user prefere...,### Input:memory management description varian...
6198,model integration description variant 205,Lofty integrates with Ollama for local model i...,### Input:model integration description varian...


In [17]:
#convert to huggging face dataset
#for better model understanding
dataset = Dataset.from_pandas(df[['fulltext']])

In [21]:
#Loading model and Tokenzier
model_folder = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_folder)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

C:\Users\ddeba\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ddeba\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [24]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [23]:
model = AutoModelForCausalLM.from_pretrained(model_folder)

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [26]:
#prepare for  LoRa
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=2,
    lora_alpha=4,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model,lora_config)
model.print_trainable_parameters()
model = model.to("cpu")

trainable params: 270,336 || all params: 494,303,104 || trainable%: 0.0547


In [27]:
#tokenize dataset
def tokenize_function(examples):
    return tokenizer(
        examples['fulltext'],
        truncation=True,
        max_length=100
    )
    

In [29]:
##tokenized dataset
tokenized_dataset = dataset.map(tokenize_function,batched=True,remove_columns=['fulltext'])


Map:   0%|          | 0/6200 [00:00<?, ? examples/s]

In [30]:
#data collator for causal language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [31]:
#training arguments
training_args = TrainingArguments(
    output_dir="./qwen_lofty",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    fp16=False,
    dataloader_pin_memory=False,
    use_cpu=True,
    report_to="none"
)

In [32]:
#trainer
trainer = Trainer(
    model = model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)


In [ ]:
trainer.train()

Step,Training Loss
5,4.537925
10,4.853807
15,4.716642
20,4.646706
25,4.202976
30,3.924345
35,4.281533
40,3.990138
45,3.704354
50,3.153614
